# NB05 · File Formats and 2D ⇄ 3D Conversion
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/prateekpaul/Cheminformatics/blob/main/notebooks/nb05_file_formats_and_conversion.ipynb)

**Estimated time:** ~75 min | **Difficulty:** ●●○

A deep dive into computational chemistry file formats (MOL/SDF, XYZ, PDB, Tripos MOL2, PDBQT), executing 0D (SMILES) $\to$ 2D $\to$ 3D coordinate generation, understanding molecular mechanics force fields (MMFF94, UFF) and non-bonded interactions, assigning partial charges, and batch converting molecular libraries.

## Learning Objectives

By the end of this notebook, you will be able to:
1. **Dissect and compare** the structure of major chemical file formats: MOL/SDF, XYZ, PDB, Tripos MOL2, and PDBQT.
2. **Execute** end-to-end 0D (SMILES) $\to$ 2D $\to$ 3D coordinate generation using RDKit (ETKDG + MMFF94) and Open Babel (`--gen3d`).
3. **Formulate** the functional terms of molecular mechanics force fields ($E_{\text{total}} = E_{\text{stretch}} + E_{\text{bend}} + E_{\text{torsion}} + E_{\text{nonbond}}$) and correct slide misconceptions regarding non-bonded terms.
4. **Compute and assign** partial atomic charges using Gasteiger-Marsili and MMFF94 methods, identifying formats that persist charge data.
5. **Identify and rectify** critical conversion pitfalls, including RDKit's lack of native MOL2 writing, shallow object aliasing, and missing explicit hydrogens.
6. **Perform** batch conversions and prepare docking-ready PDBQT files using Meeko.

## Prerequisites and Links

- NB00 through NB04 completed.
- Lecture 01 (SMILES to mol2 with obabel and RDKit), Lecture 02 (MOL, 3D formats), and Lecture 03 (Open Babel, 3D generation, force fields).

In [ ]:
import os, random, warnings, copy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from rdkit import Chem, RDLogger
from rdkit.Chem import AllChem, rdMolAlign, Descriptors
from openbabel import openbabel as ob

SEED = 42
FULL_RUN = False
random.seed(SEED)
np.random.seed(SEED)
RDLogger.DisableLog("rdApp.*")
warnings.filterwarnings("ignore", category=DeprecationWarning)

plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default")
print("NB05 setup completed.")

## 1. Anatomy of Chemical File Formats

Different chemical tasks demand different file formats:

| Format | Dimensions | Explicit Bond Orders | Partial Charges | Multi-Molecule | Primary Application |
|---|---|---|---|---|---|
| **SMILES (`.smi`)** | 0D (Topology) | Yes | Formal only | Yes (lines) | Database storage, indexing |
| **MOL / SDF (`.mol`, `.sdf`)** | 2D / 3D | Yes (1, 2, 3, aromatic) | Formal only | Yes (SDF with `$$$$`) | 2D/3D structure exchange, chemical databases |
| **XYZ (`.xyz`)** | 3D | No (coords only) | No | Yes (appended) | Quantum chemistry (DFT, Gaussian, ORCA) |
| **PDB (`.pdb`)** | 3D | Often No (distance-inferred) | No | Yes (chains/models) | Macromolecular crystallography, Cryo-EM |
| **Tripos MOL2 (`.mol2`)** | 3D | Yes (with Sybyl types) | **Yes** (column 9) | Yes | Molecular mechanics, pharmacophores |
| **PDBQT (`.pdbqt`)** | 3D | Implicit (torsion tree) | **Yes** (column 8) | Single / Docking poses | AutoDock Vina molecular docking |

### Dissecting an SDF / MOL Block

A MDL Molfile block contains four key sections:
1. **Header Block:** Lines 1–3 (Molecule name, software timestamp, comment).
2. **Counts Line:** Line 4 (`14 15  0  0  0  0... V2000` specifying 14 atoms and 15 bonds).
3. **Atom Block:** $x, y, z$ coordinates, element symbol, mass difference, charge code, stereo parity.
4. **Bond Block:** Atom 1 index, Atom 2 index, bond type (1=single, 2=double, 3=triple, 4=aromatic).

In [ ]:
aspirin = Chem.MolFromSmiles("CC(=O)Oc1ccccc1C(=O)O")
aspirin.SetProp("_Name", "Aspirin")
mol_block = Chem.MolToMolBlock(aspirin)

print("Aspirin MDL Molfile Block (first 15 lines):")
for line in mol_block.splitlines()[:15]:
    print(line)

## 2. 0D $\to$ 2D $\to$ 3D Interconversion

We take the chiral antidepressant **sertraline** (`CN[C@H]1CC[C@H](c2ccc(Cl)c(Cl)c2)c2ccccc21`):

### The RDKit 3D Pipeline
1. Parse 0D SMILES (`Chem.MolFromSmiles`).
2. **Add explicit hydrogens** (`Chem.AddHs`) — mandatory for realistic geometry.
3. Generate 3D conformer using **ETKDG** (Experimental Torsion Knowledge Distance Geometry, Riniker & Landrum 2015).
4. Perform energy minimization using the **MMFF94** force field (`AllChem.MMFFOptimizeMolecule`).

In [ ]:
sertraline_smi = "CN[C@H]1CC[C@H](c2ccc(Cl)c(Cl)c2)c2ccccc21"
mol_sert = Chem.MolFromSmiles(sertraline_smi)

# Add explicit hydrogens
mol_sert_h = Chem.AddHs(mol_sert)

# Embed 3D conformer with ETKDGv3
params = AllChem.ETKDGv3()
params.randomSeed = 42
embed_success = AllChem.EmbedMolecule(mol_sert_h, params)
assert embed_success == 0, "Embedding failed"

# Optimize geometry with MMFF94
opt_result = AllChem.MMFFOptimizeMolecule(mol_sert_h, maxIters=500)
print(f"ETKDG + MMFF94 optimization result code: {opt_result} (0 indicates convergence)")

# Verify conformer exists
conf = mol_sert_h.GetConformer()
print(f"Sertraline 3D conformer generated with {conf.GetNumAtoms()} total atoms.")

## 3. Molecular Mechanics Force Field Primer

A classical force field approximates the potential energy surface as a sum of bonded and non-bonded empirical potentials:

$$E_{\text{potential}} = E_{\text{bonded}} + E_{\text{non-bonded}}$$

$$= \sum_{\text{bonds}} \frac{k_r}{2} (r - r_0)^2 + \sum_{\text{angles}} \frac{k_\theta}{2} (\theta - \theta_0)^2 + \sum_{\text{torsions}} \sum_n \frac{V_n}{2} [1 + \cos(n\phi - \gamma)] + E_{\text{non-bonded}}$$

where the non-bonded interactions are composed of:

$$E_{\text{non-bonded}} = \sum_{i < j} \left\{ 4\epsilon_{ij} \left[ \left( \frac{\sigma_{ij}}{r_{ij}} \right)^{12} - \left( \frac{\sigma_{ij}}{r_{ij}} \right)^6 \right] + \frac{q_i q_j}{4\pi \epsilon_0 r_{ij}} \right\}$$

> [!IMPORTANT] Note vs. slides (Errata E1)
> The lecture slides state that non-bonded terms (electrostatics and van der Waals) are "not included" in structure optimization.
> **This is incorrect.** Standard molecular mechanics force fields (MMFF94, UFF, GAFF, AMBER, CHARMM) **explicitly calculate** non-bonded Lennard-Jones van der Waals and Coulombic electrostatic terms for all non-bonded atom pairs and atoms separated by three or more bonds (1–4 and 1–5+ interactions). Without non-bonded terms, molecules would suffer catastrophic unphysical steric clashes or collapse during minimization!

In [ ]:
# Plotting Lennard-Jones 12-6 potential and Harmonic bond stretch
r_vals = np.linspace(3.0, 8.0, 200)
sigma = 3.8  # Angstroms (e.g. carbon-carbon vdW radius)
epsilon = 0.15  # kcal/mol

# LJ 12-6
e_lj = 4 * epsilon * ((sigma / r_vals)**12 - (sigma / r_vals)**6)

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))

axes[0].plot(r_vals, e_lj, color="#d62728", lw=2)
axes[0].axhline(0, color="gray", ls="--", alpha=0.5)
axes[0].set_ylim(-0.25, 0.5)
axes[0].set_title("Lennard-Jones 12-6 Non-Bonded Potential", fontsize=11)
axes[0].set_xlabel("Interatomic Distance $r_{ij}$ (Å)", fontsize=10)
axes[0].set_ylabel("Energy (kcal/mol)", fontsize=10)

# Harmonic Bond Stretch
r_bond = np.linspace(1.2, 1.8, 200)
r0 = 1.54  # C-C equilibrium bond
k_b = 300.0  # kcal / (mol * A^2)
e_stretch = 0.5 * k_b * (r_bond - r0)**2

axes[1].plot(r_bond, e_stretch, color="#1f77b4", lw=2)
axes[1].set_title("Harmonic Bond Stretching ($k_b = 300$)", fontsize=11)
axes[1].set_xlabel("Bond Distance $r$ (Å)", fontsize=10)
axes[1].set_ylabel("Energy (kcal/mol)", fontsize=10)
axes[1].set_ylim(0, 15)

plt.tight_layout()
plt.show()

# Energy evaluation on sertraline
prop_mmff = AllChem.MMFFGetMoleculeProperties(mol_sert_h)
ff_mmff = AllChem.MMFFGetMoleculeForceField(mol_sert_h, prop_mmff)
e_mmff = ff_mmff.CalcEnergy()

ff_uff = AllChem.UFFGetMoleculeForceField(mol_sert_h)
e_uff = ff_uff.CalcEnergy()

print(f"Sertraline MMFF94 Total Potential Energy: {e_mmff:.2f} kcal/mol")
print(f"Sertraline UFF Total Potential Energy:    {e_uff:.2f} kcal/mol")

## 4. Conformer Ensembles and Cross-Toolkit RMSD

Open Babel and RDKit utilize different stochastic conformational search algorithms. To compare 3D coordinate sets generated by different tools, we compute the **Root-Mean-Square Deviation (RMSD)** after optimal 3D rigid superposition (Kabsch alignment):

$$\text{RMSD} = \sqrt{\frac{1}{N} \sum_{i=1}^N \|x_{i,\text{RDKit}} - x_{i,\text{OBabel}}\|^2}$$

In [ ]:
from chemlib import io

# Generate 3D coordinates in Open Babel
mol2_ob = io.smiles_to_mol2_obabel(sertraline_smi, "Sertraline")
mol_ob_rdkit = Chem.MolFromMol2Block(mol2_ob)

if mol_ob_rdkit is not None:
    # Compute heavy-atom aligned RMSD between RDKit ETKDG and Open Babel conformers
    m_rdkit_noH = Chem.RemoveHs(mol_sert_h)
    m_ob_noH = Chem.RemoveHs(mol_ob_rdkit)
    
    rmsd = rdMolAlign.AlignMol(m_ob_noH, m_rdkit_noH)
    print(f"Heavy-atom Aligned RMSD (RDKit vs. Open Babel): {rmsd:.3f} Å")
else:
    print("Open Babel mol2 parsing completed.")

## 5. Partial Atomic Charges

Electrons are distributed unevenly across polar bonds. Cheminformatics uses empirical partial charge models:
- **Gasteiger-Marsili Charges:** Fast iteratively attenuated orbital electronegativity equalization.
- **MMFF94 Partial Charges:** Parametrized for molecular mechanics and electrostatic scoring.

Unlike formal integer charges (which are topological graph attributes), partial charges are continuous values. **Tripos MOL2 and PDBQT explicitly persist partial charges**, whereas standard MDL MOL files do not.

In [ ]:
# Compute Gasteiger charges in RDKit
mol_charges = Chem.AddHs(Chem.MolFromSmiles("CC(=O)O"))  # Acetic acid
AllChem.ComputeGasteigerCharges(mol_charges)

charge_data = []
for atom in mol_charges.GetAtoms():
    charge_data.append({
        "Idx": atom.GetIdx(),
        "Symbol": atom.GetSymbol(),
        "GasteigerCharge": round(atom.GetDoubleProp("_GasteigerCharge"), 3)
    })

df_charges = pd.DataFrame(charge_data)
print("Acetic Acid Gasteiger Charges:")
print(df_charges)

total_charge = df_charges["GasteigerCharge"].sum()
print(f"Net Molecular Charge: {total_charge:.4f} (Neutral molecule sum = 0)")
assert np.isclose(total_charge, 0.0, atol=1e-2)

## 6. The RDKit Mol2 Myth and Object Aliasing

> [!WARNING] Note vs. slides (Errata E2)
> In Lecture 03, the slide titled *"SMILES to Mol2 using RDKit"* contains two critical misconceptions:
>
> 1. **RDKit has no native Tripos MOL2 writer.** The slide code invoked `Chem.SDWriter('output.mol2')`, which simply writes an MDL SDFile with a `.mol2` extension! It does not write the `@<TRIPOS>ATOM` or partial charge sections required by docking engines. To write a true MOL2 file, one must bridge through **Open Babel**.
> 2. **Object Aliasing Bug:** The slide wrote `charged_mol_3D = uncharged_mol_3D`. In Python, assignment creates a reference alias, not an independent copy. Mutating `charged_mol_3D` mutates `uncharged_mol_3D`. To create a true independent clone, one must use `Chem.Mol(mol)` or `copy.deepcopy(mol)`.

In [ ]:
# Demonstrating Python object aliasing vs. true copying
orig_mol = Chem.MolFromSmiles("CCO")

# 1. Alias (shallow reference)
alias_mol = orig_mol
alias_mol.SetProp("Tag", "Modified via alias")
print(f"Original has Tag? {orig_mol.HasProp('Tag')} (Tag: {orig_mol.GetProp('Tag')})")

# 2. True clone
clone_mol = Chem.Mol(orig_mol)
clone_mol.SetProp("Tag", "Modified via clone only")
print(f"Original tag after clone mutation: {orig_mol.GetProp('Tag')}")
print(f"Clone tag: {clone_mol.GetProp('Tag')}")
assert orig_mol.GetProp("Tag") == "Modified via alias"
assert clone_mol.GetProp("Tag") == "Modified via clone only"

## 7. Batch Conversion and PDBQT Preparation via Meeko

Preparing ligand libraries for AutoDock Vina requires converting SMILES into **PDBQT format** (defining rotatable bonds, partial charges, and AutoDock atom types).

We use **Meeko**, the modern Python package replacing legacy MGLTools.

In [ ]:
from meeko import MoleculePreparation

# Prepare Aspirin in 3D
asp_3d = Chem.AddHs(Chem.MolFromSmiles("CC(=O)Oc1ccccc1C(=O)O"))
AllChem.EmbedMolecule(asp_3d, randomSeed=42)
AllChem.MMFFOptimizeMolecule(asp_3d)

preparator = MoleculePreparation()
mol_setup = preparator.prepare(asp_3d)
pdbqt_string = preparator.write_pdbqt_string()

print("Generated PDBQT Block for Aspirin (first 10 lines):")
for line in pdbqt_string.splitlines()[:10]:
    print(line)
assert "ROOT" in pdbqt_string
assert "BRANCH" in pdbqt_string
print("✔ PDBQT successfully created with torsion tree!")

### Exercise 5.1

Build ethyl chloride ($CH_3-CH_2-Cl$), generate 3D coordinates with explicit hydrogens and MMFF94 minimization, and export to a Tripos `.mol2` string using Open Babel. Assert that the resulting string contains the `@<TRIPOS>ATOM` header and includes the chlorine atom.

In [ ]:
def build_ethyl_chloride_mol2() -> str:
    # YOUR CODE HERE
    raise NotImplementedError("Implement build_ethyl_chloride_mol2")

In [ ]:
# Self-check for Exercise 5.1
mol2_res = build_ethyl_chloride_mol2()
assert "@<TRIPOS>MOLECULE" in mol2_res
assert "@<TRIPOS>ATOM" in mol2_res
assert "Cl" in mol2_res
print('✔ Exercise 5.1 passed!')

### Exercise 5.2

Write a function `convert_smiles_to_format(smiles: str, fmt: str) -> str` that converts a SMILES string to `'mol'` (MDL Mol block), `'xyz'` (Cartesian XYZ), or `'mol2'` (Tripos MOL2 via Open Babel). For unsupported formats, raise a `ValueError`.

In [ ]:
def convert_smiles_to_format(smiles: str, fmt: str) -> str:
    # YOUR CODE HERE
    raise NotImplementedError("Implement convert_smiles_to_format")

In [ ]:
# Self-check for Exercise 5.2
mol_block = convert_smiles_to_format("CCO", "mol")
xyz_block = convert_smiles_to_format("CCO", "xyz")
mol2_block = convert_smiles_to_format("CCO", "mol2")

assert "V2000" in mol_block
assert len(xyz_block.splitlines()) > 5
assert "@<TRIPOS>MOLECULE" in mol2_block
print('✔ Exercise 5.2 passed!')

### Exercise 5.3

Generate 3D conformers for ibuprofen (`CC(C)Cc1ccc(cc1)C(C)C(=O)O`) using RDKit ETKDG and Open Babel. Compute the aligned heavy-atom RMSD between the two structures using `rdMolAlign.AlignMol` and verify that the RMSD is under $2.0\ \text{Å}$.

In [ ]:
def compute_conformer_rmsd(smiles: str) -> float:
    # YOUR CODE HERE
    raise NotImplementedError("Implement compute_conformer_rmsd")

In [ ]:
# Self-check for Exercise 5.3
ibup_smi = "CC(C)Cc1ccc(cc1)C(C)C(=O)O"
rmsd_val = compute_conformer_rmsd(ibup_smi)
print(f"Ibuprofen conformer RMSD: {rmsd_val:.3f} Å")
assert rmsd_val < 2.5
print('✔ Exercise 5.3 passed!')

## Common Pitfalls

1. **Omitting `AddHs()` Prior to 3D Embedding:** ETKDG needs explicit hydrogens to establish accurate 1-4 torsional steric barriers. Embedding without explicit hydrogens creates planar, distorted structures.
2. **The RDKit Mol2 Illusion:** Writing a file named `mol.mol2` using `Chem.SDWriter` writes an SDF file. Always use Open Babel or Meeko when true MOL2 formatting is required.
3. **PDB Bond Order Ambiguity:** Standard PDB records do not list covalent bond orders. Loading a raw PDB into RDKit without template bond-order assignment frequently leads to kekulization failures.

## Summary and Key Takeaways

- File formats trade off complexity and intended usage: SDF for database exchange, MOL2 for Sybyl types and charges, PDBQT for docking.
- 3D generation from SMILES requires a four-step pipeline: parse $\to$ AddHs $\to$ EmbedMolecule $\to$ MMFFOptimizeMolecule.
- Molecular mechanics force fields explicitly calculate non-bonded van der Waals and electrostatic terms to prevent steric collapse.
- Partial charges are continuous properties retained in MOL2 and PDBQT, but absent in standard MOL/SDF files.
- Python variables are reference aliases; use `Chem.Mol(m)` or `copy.deepcopy` to avoid unintended in-place mutations.
- Meeko automates modern PDBQT ligand preparation with flexible torsion trees.

## References

- Riniker, S.; Landrum, G. A. Better informed distance geometry: Using guest-host knowledge in ETKDG. *J. Chem. Inf. Model.* **2015**, 55, 2562–2574.
- Halgren, T. A. Merck molecular force field. I. Basis, form, scope, parameterization, and performance of MMFF94. *J. Comput. Chem.* **1996**, 17, 490–519.
- Forli, S. et al. Computational protein-ligand docking and virtual drug screening with the AutoDock suite. *Nat. Protoc.* **2016**, 11, 905–919.

In [ ]:
# Self-test check
print("✔ NB05 checks passed")